In [1]:
# install the required dependecies
# !pip install beautifulsoup4
# !pip install requests

In [2]:
# import the required libraries
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

In [3]:
# data collection
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

res = []

for i in range(40):
    url = f'https://www.carlist.my/cars-for-sale/malaysia?inspected=true&page_number={i}&page_size=25'
    html = requests.get(url, headers=headers).text
    soup = BeautifulSoup(html, 'html.parser')
    cars = soup.find_all('div', {'class':'grid grid--full cf'})

    for car in cars:
        description = car.find('a', {'class': 'ellipsize js-ellipsize-text'}).text.strip()
        monthly_installment = car.find('div', {'class': 'listing__installment'}).text.strip()
        
        # Check if listing price exists
        listing_price_elem = car.find('div', {'class': 'listing__price delta weight--bold'})
        if listing_price_elem:
            listing_price = listing_price_elem.text.strip()
        else:
            listing_price = "N/A"  # or any default value you prefer
        
        # Check if model exists
        rating_div = car.find('div', {'class': 'listing__rating'})
        if rating_div:
            model_elem = rating_div.find('div', {'class': 'listing__rating-model'})
            if model_elem:
                model = model_elem.text.strip()
            else:
                model = "N/A"
        else:
            model = "N/A"
        
        spec_elem = car.find('div', {'class': 'listing__specs'}).find('div', {'class': 'item'})
        if spec_elem:
            spec_text = spec_elem.text.strip()
        else:
            spec_text = "N/A"
        
        # Check if gear and location exist
        gear_elem = car.find('div', {'class': 'listing__specs'}).find_all('div', {'class': 'item'})[1]
        if gear_elem:
            gear = gear_elem.text.strip()
        else:
            gear = "N/A"
        
        location_elem = car.find('div', {'class': 'listing__specs'}).find_all('div', {'class': 'item'})[3]
        if location_elem:
            location = location_elem.text.strip()
        else:
            location = "N/A"
        
        res_list = [description, monthly_installment, listing_price, model, spec_text, gear, location]
        res.append(res_list)


In [4]:
# save the collected data in pandas dataframe format
df = pd.DataFrame(res)
df.columns = ['Description','Monthly_Installment','List_Price','Model','Milleage','Gear_Type','Location']

In [5]:
# view the first 5 rows of the dataframe
df.head()

,Description,Monthly_Installment,List_Price,Model,Milleage,Gear_Type,Location
0,2013/2014 BMW 528i 2.0 Sedan - M Sport service...,"RM 1,139 / month","RM 87,888",BMW 528i,145 - 150K KM,Automatic,"Selangor, Puchong"
1,2017 Honda Civic 1.8 S i-VTEC Sedan - (A) NO P...,RM 918 / month,"RM 70,800",Honda Civic,100 - 105K KM,Automatic,"Kuala Lumpur, Cheras"
2,2012 BMW 328i 2.0 Luxury Line Sedan - Luxury (...,RM 594 / month,"RM 45,800",BMW 328i,135 - 140K KM,Automatic,"Kuala Lumpur, Cheras"
3,2014 Toyota Corolla Altis 2.0 V Sedan - (A) NO...,RM 646 / month,"RM 49,800",Toyota Corolla Altis,120 - 125K KM,Automatic,"Kuala Lumpur, Cheras"
4,2013 Honda Accord 2.4 i-VTEC VTi-L Sedan - VTi...,RM 594 / month,"RM 45,800",Honda Accord,120 - 125K KM,Automatic,"Kuala Lumpur, Cheras"


In [6]:
# export the dataframe in csv file
# df.to_csv('resale_car.csv', index=False)